# Northwind Ambient Ops — EDA
Step 1: Profile each table before touching any queries.

**Rule:** understand the data first, fix SQL second.

In [ ]:
import duckdb
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

con = duckdb.connect()

tables = {
    'note':         'data/note.csv',
    'note_audit':   'data/note_audit.csv',
    'clinician':    'data/clinician.csv',
    'mds':          'data/mds.csv',
    'escalation':   'data/escalation.csv',
    'sla_config':   'data/sla_config.csv',
    'rubric_weight':'data/rubric_weight.csv',
}
for name, path in tables.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_csv_auto('{path}')")

print('Tables loaded.')

## Table 1 — `note`
**Expected grain:** one row per note. **Actual grain:** one row per *ingestion* of a note.

In [ ]:
print('--- Row counts ---')
display(con.execute("""
    SELECT
        COUNT(*)                    AS total_rows,
        COUNT(DISTINCT note_id)     AS unique_notes,
        COUNT(*) - COUNT(DISTINCT note_id) AS duplicate_rows
    FROM note
""").df())

print('--- Voided notes ---')
display(con.execute("SELECT COUNT(*) AS voided FROM note WHERE is_void = true").df())

print('--- NULL word_count (transcript failures) ---')
display(con.execute("SELECT COUNT(*) AS null_word_count FROM note WHERE word_count IS NULL").df())

print('--- Sample duplicate note_ids ---')
display(con.execute("""
    SELECT note_id, COUNT(*) AS ingestion_count
    FROM note
    GROUP BY note_id
    HAVING COUNT(*) > 1
    ORDER BY ingestion_count DESC
    LIMIT 5
""").df())

print('--- Date range ---')
display(con.execute("SELECT MIN(submitted_at_utc) AS earliest, MAX(submitted_at_utc) AS latest FROM note").df())

## Table 2 — `clinician`
**Expected grain:** one row per doctor. **Actual grain:** one row per *version* of a doctor's record (SCD Type 2).

> If you JOIN to this table without filtering `is_current_record = true`, notes for doctors with 2 rows get counted twice.

In [ ]:
print('--- Row counts ---')
display(con.execute("""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT clinician_id) AS unique_ids
    FROM clinician
""").df())

print('--- clinician_ids with more than 1 row (SCD2 history) ---')
display(con.execute("""
    SELECT clinician_id, COUNT(*) AS row_count
    FROM clinician
    GROUP BY clinician_id
    HAVING COUNT(*) > 1
""").df())

print('--- Current vs historical records ---')
display(con.execute("SELECT is_current_record, COUNT(*) AS cnt FROM clinician GROUP BY is_current_record").df())

## Table 3 — `mds`
**Grain:** one row per specialist. Clean.

> Watch out: 2 specialists are INACTIVE. The leaderboard query (Q4) includes them.

In [ ]:
display(con.execute("SELECT status, COUNT(*) AS cnt FROM mds GROUP BY status").df())

print('--- INACTIVE specialists ---')
display(con.execute("SELECT mds_id, mds_name, status FROM mds WHERE status = 'INACTIVE'").df())

## Table 4 — `note_audit`
**Grain:** one row per audit. `note_id` is unique here — confirmed.

> Only ~28% of notes are audited. Rubric changed mid-quarter (v1 → v2 on May 15).

In [ ]:
display(con.execute("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT note_id) AS unique_notes FROM note_audit").df())

print('--- Pass / Fail split ---')
display(con.execute("SELECT pass_fail, COUNT(*) AS cnt FROM note_audit GROUP BY pass_fail").df())

print('--- Rubric version split ---')
display(con.execute("SELECT rubric_version, COUNT(*) AS cnt FROM note_audit GROUP BY rubric_version").df())

## Table 5 — `escalation`
**Grain:** one row per escalation.

> Three statuses: OPEN, RESOLVED, PENDING_POST. PENDING_POST = never posted to Slack. Nobody worked these.

In [ ]:
display(con.execute("SELECT status, COUNT(*) AS cnt FROM escalation GROUP BY status").df())

print('--- PENDING_POST detail ---')
display(con.execute("""
    SELECT
        status,
        COUNT(*) AS cnt,
        COUNT(slack_thread_ts) AS has_thread_ts,
        COUNT(last_api_error) AS has_error,
        MIN(created_at_utc) AS earliest,
        MAX(created_at_utc) AS latest
    FROM escalation
    WHERE status = 'PENDING_POST'
    GROUP BY status
""").df())

print('--- Error message ---')
display(con.execute("SELECT DISTINCT last_api_error FROM escalation WHERE status = 'PENDING_POST'").df())

## Table 6 — `sla_config`
**Grain:** one row per product line / priority / effective period.

> Targets tightened on May 15 — mid-quarter. Queries must join with date range, not just product line + priority.

In [ ]:
display(con.execute("SELECT * FROM sla_config ORDER BY product_line, priority, effective_from").df())

## Table 7 — `rubric_weight`
**Grain:** one row per rubric version per scoring dimension.

> Pass threshold: v1 = 0.85, v2 = 0.90. Changed May 15. Notes before and after are judged by different standards.

In [ ]:
display(con.execute("""
    SELECT rubric_version, effective_from, pass_threshold, COUNT(*) AS dimensions
    FROM rubric_weight
    GROUP BY rubric_version, effective_from, pass_threshold
    ORDER BY effective_from
""").df())

print('--- All weights ---')
display(con.execute("SELECT * FROM rubric_weight ORDER BY rubric_version, dimension").df())

## EDA Summary

| Table | Finding |
|---|---|
| `note` | `note_id` not unique — 62 re-ingested notes. 90 voided. 152 NULL word counts. |
| `clinician` | SCD2 — 4 clinician_ids have 2 rows. Naïve JOIN duplicates notes. |
| `mds` | 2 INACTIVE specialists included in Q4 leaderboard. |
| `note_audit` | 1,780 audits out of 6,235 notes (~28%). Rubric split: 851 v1, 929 v2. |
| `escalation` | 43 PENDING_POST — never posted to Slack, never worked. All failed with HTTP 429. |
| `sla_config` | Targets changed May 15 mid-quarter. 4 old + 4 new = 8 rows. |
| `rubric_weight` | Pass threshold raised from 0.85 (v1) to 0.90 (v2) on May 15. |